#Instructions

This code will add satellite features to your site labels using Google Earth Engine. It will produce a series of .csv files that can be used to run tests.

This code no longer requires you to upload your labels file to GEE. You simiply need a GEE project already established. You can put the link to your GEE project and the labels .csv file directly into this code.

For best results, use the first code in this process ("Create Label Data", found in Github) for generating site labels.

This code is specifically for .csv files that do not have a Date column.

Before running this code, please scroll through and identify the locations that say "USER INPUT". There are multiple spots in the second block of cdoe where you must input your own file names and this code will not run correctly if you do not update these sections.

This code works for both state and federal labels.


#Important Note!
Once this code runs, it will generate a series of .csv files that save to the highest folder of your Google Drive.

You will see these .csv files slowly populate in your G Drive folder. This code will say "complete" before those files fully populate, so after completing this code you'll have to wait a few extra minutes to make sure all files are there. This is because GEE is working in the background to generate the results. The final line of this code will tell you how many files it intends to create, so you can check when GEE is done generating files. You can also track the progress of file generation in the GEE website interface.


In [28]:
%pip install geemap

In [29]:
import geemap
import ee
import numpy as np
import os
from matplotlib import pyplot as plt
import pickle
import pandas as pd
import numpy as np
import shapely
import geopandas as gpd
import ee # earth engine
import folium
import datetime

from shapely.geometry import Point
from google.colab import drive

DEBUG = False

from pathlib import Path
import shutil
import os

drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


#User Input Section
Fill out the following two sections with your parameters anywhere that says "USER INPUT".

In [30]:
#------USER INPUT--------#
#Copy the link to your labels .csv file into the following line:

base_directory = "/content/drive/MyDrive/MOSAIKS/superfund_project_summer2026/Data"

test_labels = base_directory + "/Intermediate/mn_remediation_combinedlabels_01_2026-09-11_212.csv"

test_labels = pd.read_csv(test_labels)

display(test_labels.head(10))

,Row_Count,lat,lon,indicator
0,0,43.865,-93.345,1
1,1,43.865,-93.065,1
2,2,43.865,-93.055,1
3,3,43.865,-93.045,1
4,4,44.005,-93.185,1
5,5,44.045,-93.235,1
6,6,44.045,-93.225,1
7,7,44.055,-93.225,1
8,8,44.075,-93.225,1
9,9,44.085,-93.265,1


In [31]:
#------USER INPUT--------#
#Copy the name of your GEE project below
#The name of your GEE project can be found in the top right corner of your GEE website
ee.Authenticate()
ee.Initialize(project='superfund-featurizing') #insert GEE project name here

batch = test_labels

batch_gdf = gpd.GeoDataFrame(
    batch,
    geometry=gpd.points_from_xy(batch["lon"], batch["lat"]),
    crs="EPSG:4326")

# Convert to Earth Engine FeatureCollection
photo_fc = geemap.gdf_to_ee(batch_gdf, geodesic=True)

#------USER INPUT--------#
#Use the same information as your labels.

res: float = 0.01
location = "Minnesota"
parcel_check = False
zeroes = 1.0
suffix = "" # put a suffix for non-main workflow tests like CERCLIS



# logic based on user input; do not edit.
grid_delta = res
buff = (res*5)/10
res_string = str(res).split('.')[1]
zeroes_string = str(int(zeroes)*100)


project_file = location + "_r" + res_string + "_z" + zeroes_string + suffix


if res == 0.01:
  scale: float = 1000
elif res == 0.1:
  scale: float = 100
elif res == 0.001:
  scale: float = 10000

bands = np.arange(0,64).astype(str)
bands = ["A0" + x if len(x)==1 else "A"+x for x in bands]

# Combined reducer. Processes each band that we want to summarize over the given rectangle
reducer = (ee.Reducer.mean()
           .combine(ee.Reducer.percentile([0,10,20,30,40,50,60,70,80,90,100]), '', True))

total = len(batch_gdf)

In [32]:
points = photo_fc
points = points.toList(points.size())

if DEBUG:
    points = points.slice(0,1_000)

def make_rect(feature):
    lon = ee.Number(feature.get('lon'))
    lat = ee.Number(feature.get('lat'))

    rect = ee.Geometry.Rectangle([
        lon.subtract(buff), lat.subtract(buff),
        lon.add(buff), lat.add(buff)
    ])

    return ee.Feature(rect, feature.toDictionary())

batch_size = 500  # features per batch
if DEBUG:
    total = 1_000
num_batches = (total // batch_size) + 1

print(f"Total features: {total}, batches: {num_batches}")

today_date = datetime.date.today().strftime("%Y-%m-%d")

# Loop over batches and export each one
for i in range(num_batches):
    start = i * batch_size
    end = start + batch_size
    subset = points.slice(start,end)
    subset = ee.FeatureCollection(subset)
    fname = f'GEE_featurization_{project_file}_{i}' #PLEASE DON'T EDIT THIS. USE VARIABLES UP TO TO CONFIGURE FILE NAME


    rects = subset.map(make_rect)

    collection = (ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL')
              .filterDate("2025-01-01") #Imagery is set to a specific year
              .filterBounds(rects)
             )

    composite = collection.select(bands).median() ## Should just be one image anyway. Median over time dimension

    stats_per_rect = composite.reduceRegions(
        collection=rects,
        reducer=reducer,
        scale=scale,
        crs='EPSG:4326',
    )

    ## Start download

    task = ee.batch.Export.table.toDrive(
        collection=stats_per_rect,
        description=f'GEE_featurization_{project_file}_{i}', #PLEASE DON'T EDIT THIS. USE VARIABLES UP TO TO CONFIGURE FILE NAME
        folder=f"gee_container",
        fileNamePrefix=fname,
        fileFormat='CSV')

    task.start()
    print('Export task started:', task.status())


Total features: 44, batches: 1
Export task started: {'state': 'READY', 'description': 'GEE_featurization_Minnesota_r01_z100_0', 'priority': 100, 'creation_timestamp_ms': 1790001031887, 'update_timestamp_ms': 1790001031887, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_FEATURES', 'id': 'CYD7CHD2A7QHJFZEBVM5IXYO', 'name': 'projects/superfund-featurizing/operations/CYD7CHD2A7QHJFZEBVM5IXYO'}
